# Table of Contents
* [Magic Variables](#magic)
* [Helper Functions](#functions)
* [Load GTFs](#gtf)
* [Load Hybrids Data](#load_data)
    * [Skin Cells](#skin)
    * [Skin Pooled Cells](#pooled_skin)
    * [Wang et al Cells (6 cell types)](#wang_et_al)
    * [IPSCs and CNCCs Cells](#ipsc_cncc)
    * [Cortical Organoids](#cortical) 
    * [Endothelial cells](#endo)
    * [Cartilage Organoids](#CO)
    * [ExpLBM](#ExpLBM)
* [Combining Hybrids Data](#combining)
    * [Adding TSS info](#adding_tss)
* [Crete ASE Data](#create_ase)    
    * [Define ASE Genes](#define_ASE)
    * [Create ASE Info Table](#create_ase_table)
    * [Exporting ASE Table](#export_ase)
* [Write a Bed File for UCSC Tracks](#bed)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path
import pybedtools
from pybedtools import BedTool
import copy
import pickle
from functools import reduce
import matplotlib.pyplot as plt
from matplotlib_venn import venn2,venn3

<a id="magic"></a>
# Magic Variables

In [2]:
import_path = Path('/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Hybrids/human_chimp')
export_path = Path('/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Hybrids/human_chimp/summarized_data/')
analysis_path = Path('/home/labs/davidgo/galbo/hybrids-data/hybrids_preprocess_analysis')

<a id="functions"></a>
# Helper Functions

In [3]:
def calculate_tpm(counts, gene_lengths):
    """
    counts: a series of gene names and their counts values
    gene_lengths: a series of gene names and their lengths
    """
    aligned_counts, aligned_gene_lengths = counts.align(gene_lengths, join='left')
    rpk = aligned_counts.div(aligned_gene_lengths, axis=0)  
    per_milion_scaling_factor = (rpk.sum())/1e6  
    TPM = rpk.div(per_milion_scaling_factor)
    
    return TPM

<a id="gtf"></a>
# Load GTFs

In [4]:
with open(analysis_path/'chimp_gtf_unique_genes.txt', 'r') as file:
    chimp_gtf_unique_genes = [line.strip() for line in file]
with open(analysis_path/'human_gtf_unique_genes.txt', 'r') as file:
    human_gtf_unique_genes = [line.strip() for line in file]
gtf_genes_combined = set(human_gtf_unique_genes + chimp_gtf_unique_genes)

with open(analysis_path/'hg19_ncbi_gtf_unique_genes.txt', 'r') as file:
    hg19_ncbi_gtf_unique_genes = [line.strip() for line in file]

<a id="load_data"></a>
# Load hybrids Data

<b> From this data we will find the differentialy expressed genes (each row is a gene and the colums are different parameters for that gene) </b>

In [5]:
hybrids_dfs = {}

<a id="skin"></a>
## Skin cells

In [6]:
# 24 pseaudobulk skin organoids
skin_cell_types = ["0_Fibroblast",
                   "1_Fibroblast",
                   "2_Mesenchyme",
                   "3_Fibroblast",
                   "4_Prechondral_Mesenchyme",
                   "5_Fibroblast",
                   "6_Dermal_Papilla",
                   "7_Fibroblast_Mesenchyme_Mash",
                   "8_Also_Prechondral_Mesenchyme",
                   "9_Chondrocytes",
                   "10_Fibroblast",
                   "10_Neural_Progenitors_+_BAD",
                   "11_Weird_CNC_Fibroblast",
                   "12_Schwann_Cells",
                   "16_Melanocytes",
                   "18_Proliferating_Cells",
                   "20_Absolutely_No_Idea",
                   "21_Merkel_Cells",
                   "Cycling_Keratinocyte_Precursors",
                   "HF_Inner_Root_Sheath",
                   "HF_Outer_Root_Sheath",
                   "HF_Sebocytes",
                   "Keratinocytes",
                   "KRT6B+_Root_Sheath"]

In [7]:
skin_dfs = []
for skin_cell in skin_cell_types:
    # human alignment
    skin_h_ref = pd.read_csv(import_path/f"Skin_organoids/Pseudobulk/Humreffed/{skin_cell}_DESeq2_Humr.txt", sep = "\t")
    skin_h_ref.rename(columns= {'log2FoldChange': f"{skin_cell}_LFC_human_ref", 
                                "pvalue": f"{skin_cell}_pvalue_human_ref", 
                                "padj": f"{skin_cell}_LFC_padj_human_ref", 
                                "padj_mine": f"{skin_cell}_padj_mine_human_ref"}, inplace=True)
    #chimp alignment
    skin_c_ref = pd.read_csv(import_path/f"Skin_organoids/Pseudobulk/Chpreffed/{skin_cell}_DESeq2_chpr.txt", sep = "\t")
    skin_c_ref.rename(columns= {'log2FoldChange': f"{skin_cell}_LFC_chimp_ref", 
                                "pvalue": f"{skin_cell}_pvalue_chimp_ref", 
                                "padj": f"{skin_cell}_LFC_padj_chimp_ref", 
                                "padj_mine": f"{skin_cell}_padj_mine_chimp_ref"}, inplace=True)

    # combine them
    skin_h_c = skin_h_ref.merge(skin_c_ref, how = "outer", on = "Gene").loc[:,["Gene",
                                                                               f"{skin_cell}_LFC_human_ref",
                                                                               f"{skin_cell}_LFC_padj_human_ref",
                                                                               f"{skin_cell}_LFC_chimp_ref",
                                                                               f"{skin_cell}_LFC_padj_chimp_ref"]]
    
    # TPM values
    skin_h_c[f"{skin_cell}_TPM_total"] = None  # add this column and later fill it with real values
    skin_h_c = skin_h_c.set_index('Gene')
    
    skin_h_ref_tpm = pd.read_csv(import_path/f"Skin_organoids/Pseudobulk/Humreffed/{skin_cell}_DESeq2_Humr_tpm.txt", sep = "\t")
    skin_h_ref_tpm.insert(skin_h_ref_tpm.shape[1], f"{skin_cell}_TPM_human_allele", skin_h_ref_tpm.filter(like='_Human').mean(axis=1))
    skin_h_ref_tpm.insert(skin_h_ref_tpm.shape[1], f"{skin_cell}_TPM_chimp_allele", skin_h_ref_tpm.filter(like='_Chimp').mean(axis=1))
    
    skin_h_c = pd.merge(skin_h_c, skin_h_ref_tpm[[f"{skin_cell}_TPM_human_allele", f"{skin_cell}_TPM_chimp_allele"]], left_index=True, right_index=True, how='left')
    
    hybrids_dfs[skin_cell] = skin_h_c

In [8]:
hybrids_dfs[skin_cell].columns.tolist()

['KRT6B+_Root_Sheath_LFC_human_ref',
 'KRT6B+_Root_Sheath_LFC_padj_human_ref',
 'KRT6B+_Root_Sheath_LFC_chimp_ref',
 'KRT6B+_Root_Sheath_LFC_padj_chimp_ref',
 'KRT6B+_Root_Sheath_TPM_total',
 'KRT6B+_Root_Sheath_TPM_human_allele',
 'KRT6B+_Root_Sheath_TPM_chimp_allele']

<a id="pooled_skin"></a>
### Pooled Skin Cells 
<b>Since each cell type separately didn’t have a lot of ASE genes we asked Alex to redo the analysis with pooled clusters

In [9]:
# cell type name: [TPM_file_name, LFC_file_name]
pooled_cells = {"Pooled_fibroblasts": ["All_Fibroblast_DESeq2_Humr_tpm", "All_Fibroblast_Filtered (1)"], 
                "Pooled_epidermis": ["All_Epi_DESeq2_Humr_tpm", "All_Epi_Filtered"],
                "Pooled_mesenchyme" : ["All_Mes_DESeq2_Humr_tpm", "All_Mes_Filtered"]}

In [10]:
for name, (tpm_file_name, lfc_file_name) in pooled_cells.items():
    cell_info = pd.read_csv(import_path/f"Skin_organoids/Pseudobulk/Pooled_cell_types/{lfc_file_name}.txt", sep = "\t")
    cell_info.rename(columns={"L2FC Humreffed": f"{name}_LFC_human_ref",
                              "pvalue Humreffed": f"{name}_pvalue_human_ref", 
                              "padj Humreffed": f"{name}_LFC_padj_human_ref",
                              "padj_mine Humreffed": f"{name}_padj_mine_human_ref",
                              "L2FC Chpreffed": f"{name}_LFC_chimp_ref",
                              "pvalue Chpreffed": f"{name}_pvalue_chimp_ref", 
                              "padj Chpreffed": f"{name}_LFC_padj_chimp_ref",
                              "padj_mine Chpreffed": f"{name}_padj_mine_chimp_ref",
                              "Abs L2FC Dif": f"{name}_ABS_LFC_diff"}, inplace=True)
        
    # TPM values
    cell_info[f"{name}_TPM_total"] = None  # add this column and later fill it with real values
    cell_info = cell_info.set_index('Gene')

    skin_h_ref_tpm = pd.read_csv(import_path/f"Skin_organoids/Pseudobulk/Pooled_cell_types/{tpm_file_name}.txt", sep = "\t", index_col=0)
    skin_h_ref_tpm.insert(skin_h_ref_tpm.shape[1], f"{name}_TPM_human_allele", skin_h_ref_tpm.filter(like='_Human').mean(axis=1))
    skin_h_ref_tpm.insert(skin_h_ref_tpm.shape[1], f"{name}_TPM_chimp_allele", skin_h_ref_tpm.filter(like='_Chimp').mean(axis=1))
    cell_info = pd.merge(cell_info, skin_h_ref_tpm[[f"{name}_TPM_human_allele", f"{name}_TPM_chimp_allele"]], left_index=True, right_index=True, how='left')
    
    # skin_dfs.append(cell_info)
    hybrids_dfs[name] = cell_info[[f'{name}_LFC_human_ref', 
                                   f'{name}_LFC_padj_human_ref', 
                                   f'{name}_LFC_chimp_ref', 
                                   f'{name}_LFC_padj_chimp_ref',
                                   f'{name}_TPM_total',
                                   f"{name}_TPM_human_allele", 
                                   f"{name}_TPM_chimp_allele"]]

<b> Prechondral_Mesenchyme_Pooled </b> (added after the other pooled cells)

In [11]:
skin_cell = "Pooled_prechondral_mesenchyme"

In [12]:
skin_h_ref = pd.read_csv(import_path/f"Skin_organoids/Pseudobulk/Pooled_cell_types/Prechondral_Mesenchyme_Pooled_DESeq2_Humr.txt", sep = "\t")
skin_h_ref.rename(columns= {'log2FoldChange': f"{skin_cell}_LFC_human_ref", 
                            "pvalue": f"{skin_cell}_pvalue_human_ref", 
                            "padj": f"{skin_cell}_LFC_padj_human_ref", 
                            "padj_mine": f"{skin_cell}_padj_mine_human_ref"}, inplace=True)
#chimp alignment
skin_c_ref = pd.read_csv(import_path/f"Skin_organoids/Pseudobulk/Pooled_cell_types/Prechondral_Mesenchyme_Pooled_DESeq2_chpr.txt", sep = "\t")
skin_c_ref.rename(columns= {'log2FoldChange': f"{skin_cell}_LFC_chimp_ref", 
                            "pvalue": f"{skin_cell}_pvalue_chimp_ref", 
                            "padj": f"{skin_cell}_LFC_padj_chimp_ref", 
                            "padj_mine": f"{skin_cell}_padj_mine_chimp_ref"}, inplace=True)

# combine them
skin_h_c = skin_h_ref.merge(skin_c_ref, how = "outer", on = "Gene").loc[:,["Gene",
                                                                           f"{skin_cell}_LFC_human_ref",
                                                                           f"{skin_cell}_LFC_padj_human_ref",
                                                                           f"{skin_cell}_LFC_chimp_ref",
                                                                           f"{skin_cell}_LFC_padj_chimp_ref"]]
# TPM values
skin_h_c[f"{skin_cell}_TPM_total"] = None  # add this column and later fill it with real values
skin_h_c = skin_h_c.set_index('Gene')

skin_h_ref_tpm = pd.read_csv(import_path/f"Skin_organoids/Pseudobulk/Pooled_cell_types/Prechondral_Mesenchyme_Pooled_DESeq2_Humr_tpm.txt", sep = "\t")
skin_h_ref_tpm.insert(skin_h_ref_tpm.shape[1], f"{skin_cell}_TPM_human_allele", skin_h_ref_tpm.filter(like='_Human').mean(axis=1))
skin_h_ref_tpm.insert(skin_h_ref_tpm.shape[1], f"{skin_cell}_TPM_chimp_allele", skin_h_ref_tpm.filter(like='_Chimp').mean(axis=1))

skin_h_c = pd.merge(skin_h_c, skin_h_ref_tpm[[f"{skin_cell}_TPM_human_allele", f"{skin_cell}_TPM_chimp_allele"]], left_index=True, right_index=True, how='left')

hybrids_dfs[skin_cell] = skin_h_c

### New clustering (LimmaVoom results)

In [13]:
new_clustering_skin_cell_types = ["AllFibroblasts",
                                   "AllFollicle",
                                   "AllKeratinocytes",
                                   "APOE+_PTGDS+_lipid-associated_mesenchyme",
                                   "ATP1B3+_DPSYL2+_basal_keratinocytes",
                                   "Cycling_follicular_progenitors",
                                   "Cycling_keratinocyte_progenitors",
                                   "Dermal_condensate",
                                   "Dermal_papilla",
                                   "DPYSL2+_basal_keratinocytes",
                                   "DST+_follicular_progenitors",
                                   "ENO1-high_glycolytic_fibroblasts",
                                   "Hyacine_chondrocytes",
                                   "Hypertrophic_chondrocytes",
                                   "Immature_basal_keratinocytes",
                                   "Inner_root_sheath_cells",
                                   "KRT6A+_immature_suprabasal_keratinocytes",
                                   "KRT6B+_activated_keratinocytes",
                                   "KRT85+_follicular_progenitors",
                                   "LUM-high_ECM-producing_fibroblasts",
                                   "Melanocytes",
                                   "Merkel_cells",
                                   "NUPR1-high_stress-adapted_keratinocytes",
                                   "Outer_root_sheath_cells",
                                   "Pericytes",
                                   "POSTN+_keratinocytes",
                                   "RARRES2+_DPSYL2+_basal_keratinocytes",
                                   "RUNX2-_ENO1-high_glycolytic_mesenchyme",
                                   "RUNX2-_MMP2-high_ECM-remodeling_mesenchyme",
                                   "RUNX2+_SFRP2+_mesenchyme",
                                   "Schwann_cells"]

new_clustering_limma_path = import_path/f"Skin_organoids/Pseudobulk_Output_LimmaVoom"
new_clustering_deseq2_tpm_path = import_path/f"Skin_organoids/Pseudobulk_Output_DESeq2"

missing_tpm_cells = []

for skin_cell in new_clustering_skin_cell_types:
    # human alignment
    skin_h_ref = pd.read_csv(new_clustering_limma_path/f"LimmaVoom_{skin_cell}_Paired_Humreffed_Hybrid.txt", sep = "\t")
    skin_h_ref.rename(columns={'logFC': f"{skin_cell}_LFC_human_ref",
                               'padj': f"{skin_cell}_LFC_padj_human_ref"}, inplace=True)
    # Convert LFC from chimp vs human to human vs chimp
    skin_h_ref[f"{skin_cell}_LFC_human_ref"] = -skin_h_ref[f"{skin_cell}_LFC_human_ref"]

    # chimp alignment
    skin_c_ref = pd.read_csv(new_clustering_limma_path/f"LimmaVoom_{skin_cell}_Paired_Chpreffed_Hybrid.txt", sep = "\t")
    skin_c_ref.rename(columns={'logFC': f"{skin_cell}_LFC_chimp_ref",
                               'padj': f"{skin_cell}_LFC_padj_chimp_ref"}, inplace=True)
    # Convert LFC from chimp vs human to human vs chimp
    skin_c_ref[f"{skin_cell}_LFC_chimp_ref"] = -skin_c_ref[f"{skin_cell}_LFC_chimp_ref"]

    # combine LFC and padj
    skin_h_c = skin_h_ref.merge(skin_c_ref, how="outer", on="Gene").loc[:,["Gene",
                                                                            f"{skin_cell}_LFC_human_ref",
                                                                            f"{skin_cell}_LFC_padj_human_ref",
                                                                            f"{skin_cell}_LFC_chimp_ref",
                                                                            f"{skin_cell}_LFC_padj_chimp_ref"]]

    # Set Gene as index for merge
    skin_h_c = skin_h_c.set_index('Gene')

    # Try to load TPM values
    tpm_file_path = new_clustering_deseq2_tpm_path/f"DESeq2_{skin_cell}_Paired_Humreffed_Hybrid_tpm.txt"
    
    if tpm_file_path.exists():
        try:
            # TPM files have gene symbols in the first column (sometimes without a 'Gene' header)
            skin_tpm = pd.read_csv(tpm_file_path, sep = "\t", index_col=0)
            skin_tpm.index.name = "Gene"

            # Calculate human allele TPM mean (filter for HumanCounts columns)
            human_tpm_cols = [col for col in skin_tpm.columns if 'HumanCounts' in col]
            skin_tpm.insert(skin_tpm.shape[1], f"{skin_cell}_TPM_human_allele", skin_tpm[human_tpm_cols].mean(axis=1))

            # Calculate chimp allele TPM mean (filter for ChimpCounts columns)
            chimp_tpm_cols = [col for col in skin_tpm.columns if 'ChimpCounts' in col]
            skin_tpm.insert(skin_tpm.shape[1], f"{skin_cell}_TPM_chimp_allele", skin_tpm[chimp_tpm_cols].mean(axis=1))

            # Set TPM_total to None for now
            skin_tpm.insert(skin_tpm.shape[1], f"{skin_cell}_TPM_total", None)

            # Merge TPM values with LFC and padj
            skin_h_c = pd.merge(skin_h_c,
                                skin_tpm[[f"{skin_cell}_TPM_human_allele",
                                         f"{skin_cell}_TPM_chimp_allele",
                                         f"{skin_cell}_TPM_total"]],
                                left_index=True,
                                right_index=True,
                                how='outer')
        except Exception as e:
            print(f"Error loading TPM for {skin_cell}: {e}")
            missing_tpm_cells.append(skin_cell)
            skin_h_c[f"{skin_cell}_TPM_human_allele"] = None
            skin_h_c[f"{skin_cell}_TPM_chimp_allele"] = None
            skin_h_c[f"{skin_cell}_TPM_total"] = None
    else:
        missing_tpm_cells.append(skin_cell)
        skin_h_c[f"{skin_cell}_TPM_human_allele"] = None
        skin_h_c[f"{skin_cell}_TPM_chimp_allele"] = None
        skin_h_c[f"{skin_cell}_TPM_total"] = None

    hybrids_dfs[skin_cell] = skin_h_c

# Print summary of missing TPM data
if missing_tpm_cells:
    print(f"\n⚠️  Missing TPM data for {len(missing_tpm_cells)} cell type(s):")
    for cell in missing_tpm_cells:
        print(f"   - {cell}")
else:
    print("\n✓ All TPM files found and loaded successfully")


⚠️  Missing TPM data for 3 cell type(s):
   - AllFibroblasts
   - AllFollicle
   - AllKeratinocytes


In [14]:
# output new_clustering_skin_cell_types list to an excel file for later use
pd.DataFrame(new_clustering_skin_cell_types, columns=["Skin_Cell_Type"]).to_excel(export_path/"new_clustering_skin_cell_types.xlsx", index=False)

<a id="wang_et_al"></a>
## Wang et al 2024
1. Motor neurons <b> (MN) </b>
2. Retinal pigment epithelium <b> (RPE) </b>
3. Skeletal myocytes <b> (SKM) </b>
4. Cardiomyocytes <b> (CM) </b>
5. Hepatocyte progenitors <b> (HP) </b>
6. Pancreatic progenitors <b> (PP) </b>

In [15]:
wang_cell_types = {"MN": "Motor_neuron_(MN)",
                   "RPE": "Retinal_pigment_epithelium_(RPE)",
                   "SKM": "Skeletal_myocyte_(SKM)",
                   "CM": "Cardiomyocyte_(CM)",
                   "HP": "Hepatocyte_Progenitor_(HP)",
                   "PP": "Pancreatic_progenitor_(PP)"}

In [16]:
tpm_chimp_ref = pd.read_csv(import_path/"Myriad_For_David/TPM_Chpreffed.txt", sep="\t").rename(columns={'gene': 'Gene'}).set_index('Gene')
tpm_human_ref = pd.read_csv(import_path/"Myriad_For_David/TPM_Humreffed.txt", sep="\t").rename(columns={'gene': 'Gene'}).set_index('Gene')

In [17]:
for cell, cell_name in wang_cell_types.items():
    
    # human alignment
    wang_h_ref = pd.read_csv(import_path/f"Myriad_For_David/{cell}_aligned2human_qvalue_DESeq2.txt", sep = "\t")
    wang_h_ref.rename(columns={"gene": "Gene", 
                               "log2FoldChange": f"{cell_name}_LFC_human_ref", 
                               "pvalue": f"{cell_name}_pvalue_human_ref",
                               "padj": f"{cell_name}_LFC_padj_human_ref", 
                               "padj_mine": f"{cell_name}_padj_mine_human_ref", 
                               "qval_mine": f"{cell_name}_qval_mine_human_ref"}, inplace=True)
    
    #chimp alignment
    wang_c_ref = pd.read_csv(import_path/f"Myriad_For_David/{cell}_aligned2chimp_qvalue_DESeq2.txt", sep = "\t")
    wang_c_ref.rename(columns={"gene": "Gene", 
                               "log2FoldChange": f"{cell_name}_LFC_chimp_ref", 
                               "pvalue": f"{cell_name}_pvalue_chimp_ref",
                               "padj": f"{cell_name}_LFC_padj_chimp_ref", 
                               "padj_mine": f"{cell_name}_padj_mine_chimp_ref", 
                               "qval_mine": f"{cell_name}_qval_mine_chimp_ref"}, inplace=True)

    # combine them
    wang_h_c = wang_h_ref.merge(wang_c_ref, how = "outer", on = "Gene").loc[:,["Gene",f"{cell_name}_LFC_human_ref",
                                                                               f"{cell_name}_LFC_padj_human_ref",
                                                                               f"{cell_name}_LFC_chimp_ref",
                                                                               f"{cell_name}_LFC_padj_chimp_ref"]]
    
    # TPM values
    current_cell_tpm_human_ref = tpm_human_ref.filter(like=cell)
    tmp_human_replicate_mean = pd.DataFrame(current_cell_tpm_human_ref.filter(like="_total").mean(axis=1))
    
    tmp_human_replicate_mean.insert(tmp_human_replicate_mean.shape[1], 
                                    f"{cell}_TPM_human_allele", 
                                    current_cell_tpm_human_ref.filter(like='_hum').mean(axis=1))
    tmp_human_replicate_mean.rename(columns={f"{cell}_TPM_human_allele": f"{cell_name}_TPM_human_allele"}, inplace=True)
                                             
    tmp_human_replicate_mean.insert(tmp_human_replicate_mean.shape[1], 
                                    f"{cell}_TPM_chimp_allele", 
                                    current_cell_tpm_human_ref.filter(like='_chp').mean(axis=1))
    tmp_human_replicate_mean.rename(columns={f"{cell}_TPM_chimp_allele": f"{cell_name}_TPM_chimp_allele"}, inplace=True)
                                             
    wang_h_c = wang_h_c.merge(tmp_human_replicate_mean, how='left', on='Gene').rename(columns={0: f"{cell_name}_TPM_total"})
                                             
    hybrids_dfs[cell_name] = wang_h_c.set_index('Gene')

<a id="ipsc_cncc"></a>
## IPSCs, CNCCs (Gokhman et al 2021) 

In [18]:
ipscs_cnccs_ASE_raw = pd.read_excel(import_path /'iPSCs_and_CNCCs/iPSCs_CNCCs_ASE.xlsx', 
                                    sheet_name = 'SUMMARY', 
                                    skiprows=3)

In [19]:
column_names_mapping = {'gene': 'Gene', 
                        'chr': 'Chromosome', 
                        'start (hg38)': 'Start(hg38)', 
                        'end (hg38)': 'End(hg38)', 
                        'start (hg19)': 'Start(hg19)', 
                        'end (hg19)': 'End(hg19)', 
                        'strand': 'Strand', 
                        'ENTREZ': 'ID(ENTREZ)', 
                        'gene_biotype': 'Gene type', 
                        'maxCDS': 'maxCDS', 
                        'Ensembl (combined - Ensembl + Gene ORGANizer)': 'ID(Ensemble)', 
                        'TPM - ((reads/CDS)/libSize)*10^6 - iPS - ASE': 'iPSCs_TPM_total',
                        'log2FoldChange_ASE_iPS': 'iPSCs_LFC_human_ref',
                        'padj_ASE_iPS': 'iPSCs_LFC_padj_human_ref',
                        'padj_mine_ASE_iPS': 'iPSCs_LFC_padj_mine_human_ref',
                        'log2FoldChange_ASE_panTro5_iPS': 'iPSCs_LFC_chimp_ref',
                        'padj_ASE_panTro5_iPS': 'iPSCs_LFC_padj_chimp_ref',
                        'padj_mine_ASE_panTro5_iPS': 'iPSCs_LFC_padj_mine_chimp_ref',
                        'TPM - ((reads/CDS)/libSize)*10^6 - CNCC - ASE': 'CNCCs_TPM_total',
                        'log2FoldChange_ASE_CNCC': 'CNCCs_LFC_human_ref',
                        'padj_ASE_CNCC': 'CNCCs_LFC_padj_human_ref',
                        'padj_mine_ASE_CNCC': 'CNCCs_LFC_padj_mine_human_ref',
                        'log2FoldChange_ASE_panTro5_CNCC': 'CNCCs_LFC_chimp_ref',
                        'padj_ASE_panTro5_CNCC': 'CNCCs_LFC_padj_chimp_ref',
                        'padj_mine_ASE_panTro5_CNCC': 'CNCCs_LFC_padj_mine_chimp_ref'
                       }

In [20]:
ipscs_cnccs_ASE_raw = ipscs_cnccs_ASE_raw.rename(columns=column_names_mapping)[column_names_mapping.values()]

In [21]:
cds_length = ipscs_cnccs_ASE_raw.set_index('Gene')['maxCDS']
ipscs_cnccs_ASE_raw = ipscs_cnccs_ASE_raw.drop(columns=['maxCDS'])

In [22]:
ipscs_cnccs_ASE_to_use = ipscs_cnccs_ASE_raw.copy()

In [23]:
#remove data of chr20 genes from iPScs columns (since chr20 has aneuploidy)
chr20_ipscs_mask = ipscs_cnccs_ASE_to_use.columns.str.contains("iPSCs")
ipscs_cnccs_ASE_to_use.loc[ipscs_cnccs_ASE_to_use['Chromosome'] == 'chr20', chr20_ipscs_mask] = np.nan

In [24]:
ipscs_cnccs_ASE__to_use = ipscs_cnccs_ASE_to_use.loc[:,["Gene",
                                                        "CNCCs_LFC_human_ref",
                                                        "CNCCs_LFC_padj_human_ref",
                                                        "CNCCs_LFC_chimp_ref",
                                                        "CNCCs_LFC_padj_chimp_ref", 
                                                        "CNCCs_TPM_total",
                                                        "iPSCs_LFC_human_ref",
                                                        "iPSCs_LFC_padj_human_ref",
                                                        "iPSCs_LFC_chimp_ref",
                                                        "iPSCs_LFC_padj_chimp_ref", 
                                                        "iPSCs_TPM_total"]]

In [25]:
for cell in ['iPSCs', 'CNCCs']:
    ipscs_cnccs_ASE__to_use.insert(ipscs_cnccs_ASE__to_use.shape[1], f"{cell}_TPM_human_allele", None)
    ipscs_cnccs_ASE__to_use.insert(ipscs_cnccs_ASE__to_use.shape[1], f"{cell}_TPM_chimp_allele", None)

In [26]:
for cell in ['iPSCs', 'CNCCs']:
    hybrids_dfs[cell] = ipscs_cnccs_ASE__to_use.set_index('Gene').filter(like=cell)

In [27]:
# #check hg19 gtf
# ipscs_cnccs_not_in_hg19 = set(ipscs_cnccs_ASE_raw['Gene']) - set(hg19_ncbi_gtf_unique_genes)
# with open(analysis_path/f'ipscs_cnccs_genes_not_in_hg19.txt', 'w') as file:
#     # Write each item in the list to the file
#     for item in ipscs_cnccs_not_in_hg19:
#         file.write(f"{item}\n")

<a id="cortical"></a>
## Cortical Organoids (Agogila 2021)

In [28]:
# Days to process
days = ["D50", "D100", "D150"]

for day in days:
    # Read and filter the brain ASE raw data for the specific day
    brain_ASE_raw = pd.read_excel(import_path / 'Brain_organoids/NIHMS1765616-supplement-supp_tables.xlsx', 
                                  sheet_name='TableS11', 
                                  skiprows=2).set_index('Gene').filter(like=day)

    # Rename columns for the specific day
    brain_ASE_raw.rename(columns={
        f"{day}_HumanLFC": f"Cortical_organoids_{day.lower()}_LFC_human_ref",
        f"{day}_HumanPadj": f"Cortical_organoids_{day.lower()}_LFC_padj_human_ref",
        f"{day}_ChimpLFC": f"Cortical_organoids_{day.lower()}_LFC_chimp_ref",
        f"{day}_ChimpPadj": f"Cortical_organoids_{day.lower()}_LFC_padj_chimp_ref"
    }, inplace=True)

    # Read and filter the read counts data for the specific day
    h_ref_counts_df = pd.read_csv(import_path / 'Brain_organoids/Brain_organoids_read_count/hyCS.GRCh38.txt', sep='\t')
    h_ref_day_counts_df = h_ref_counts_df.filter(like=day)
    h_ref_day_counts_df.insert(0, 'Gene', ipscs_cnccs_ASE_raw['Gene'])
    h_ref_day_counts_df = h_ref_day_counts_df.set_index('Gene')

    # Calculate TPM values
    tpm_df = h_ref_day_counts_df.apply(lambda col: calculate_tpm(col, cds_length), axis=0)

    # Insert TPM columns into brain ASE raw data
    brain_ASE_raw.insert(brain_ASE_raw.shape[1], 
                         f'Cortical_organoids_{day.lower()}_TPM_total',
                         tpm_df.filter(like="total").mean(axis=1).loc[brain_ASE_raw.index])
    brain_ASE_raw.insert(brain_ASE_raw.shape[1], 
                         f'Cortical_organoids_{day.lower()}_TPM_human_allele',
                         tpm_df.filter(like="human").mean(axis=1).loc[brain_ASE_raw.index])
    brain_ASE_raw.insert(brain_ASE_raw.shape[1], 
                         f'Cortical_organoids_{day.lower()}_TPM_chimp_allele',
                         tpm_df.filter(like="chimp").mean(axis=1).loc[brain_ASE_raw.index])

    # Save to hybrids_dfs dictionary
    hybrids_dfs[f'Cortical_organoids_{day.lower()}'] = brain_ASE_raw

In [29]:
#### OLD (only d100) ####
# brain_ASE_raw = pd.read_excel(import_path/'Brain organoids/NIHMS1765616-supplement-supp_tables.xlsx', 
#                              sheet_name = 'TableS11', 
#                              skiprows=2).set_index('Gene').filter(like='D100')

# brain_ASE_raw.rename(columns={"D100_HumanLFC": "Cortical_organoids_d100_LFC_human_ref", 
#                               "D100_HumanPadj": "Cortical_organoids_d100_LFC_padj_human_ref", 
#                               "D100_ChimpLFC": "Cortical_organoids_d100_LFC_chimp_ref", 
#                               "D100_ChimpPadj": "Cortical_organoids_d100_LFC_padj_chimp_ref"}, inplace=True)

# #counts values
# h_ref_counts_df = pd.read_csv(import_path/'Brain organoids/Brain organoids read count/hyCS.GRCh38.txt', sep='\t')
# h_ref_d100_counts_df = h_ref_counts_df.filter(like='D100')
# h_ref_d100_counts_df.insert(0, 'Gene', ipscs_cnccs_ASE_raw['Gene'])
# h_ref_d100_counts_df = h_ref_d100_counts_df.set_index('Gene')

# tpm_df = h_ref_d100_counts_df.apply(lambda col: calculate_tpm(col, cds_length), axis=0)

# brain_ASE_raw.insert(brain_ASE_raw.shape[1], 'Cortical_organoids_d100_TPM_total', tpm_df.filter(like="total").mean(axis=1).loc[brain_ASE_raw.index])
# brain_ASE_raw.insert(brain_ASE_raw.shape[1], 'Cortical_organoids_d100_TPM_human_allele', tpm_df.filter(like="human").mean(axis=1).loc[brain_ASE_raw.index])
# brain_ASE_raw.insert(brain_ASE_raw.shape[1], 'Cortical_organoids_d100_TPM_chimp_allele', tpm_df.filter(like="chimp").mean(axis=1).loc[brain_ASE_raw.index])

# hybrids_dfs['Cortical_organoids_d100'] = brain_ASE_raw

<a id="endo"></a>
## Endothelial Cells

In [30]:
# cell type name: [total_tpm_column_indicator, allele_TPM_file_name, LFC_file_name]
endo_cells = {"Endothelial_not_treated": ["NT", 
                                          "DESeq2_Humreffed_NT_Human_Chimp_tpm", 
                                          "NT_Filtered"], 
              "Endothelial_TNF": ["TNF", 
                                  "DESeq2_Humreffed_TNF_Human_Chimp_tpm", 
                                  "TNF_Filtered"],
              "Endothelial_Spin" : ["Spin",
                                    "DESeq2_Humreffed_Spin_Human_Chimp_tpm", 
                                    "Spin_Filtered"]}

In [31]:
endothelial_total_TPM = pd.read_csv(import_path/f"Endothelial/Human_Referenced_Endothelial_TotalTPM.txt", sep='\t', index_col=0)

In [32]:
# padj are actually padj_mine
for name, (total_tpm_column_indicator, allele_tpm_file_name, lfc_file_name) in endo_cells.items():
    cell_info = pd.read_csv(import_path/f"Endothelial/{lfc_file_name}.txt", sep = "\t")
    cell_info.rename(columns={"log2FoldChange_Humreffed": f"{name}_LFC_human_ref",
                              "pvalue_Humreffed": f"{name}_pvalue_human_ref", 
                              "padj_mine_Humreffed": f"{name}_LFC_padj_human_ref",
                              "log2FoldChange_Chpreffed": f"{name}_LFC_chimp_ref",
                              "pvalue_Chpreffed": f"{name}_pvalue_chimp_ref", 
                              "padj_mine_Chpreffed": f"{name}_LFC_padj_chimp_ref",
                              "lfc dif": f"{name}_LFC_diff"}, inplace=True)
    
    #add allele TPMs
    cell_info = cell_info.set_index('Gene')
    tpm_cell = pd.read_csv(import_path/f"Endothelial/{allele_tpm_file_name}.txt", sep = "\t")
    tpm_cell.insert(tpm_cell.shape[1], 
                    f"{name}_TPM_human_allele", 
                    tpm_cell.filter(like='Human').mean(axis=1))
    tpm_cell.insert(tpm_cell.shape[1], 
                    f"{name}_TPM_chimp_allele", 
                    tpm_cell.filter(like='Chimp').mean(axis=1))
    cell_info = pd.merge(cell_info, 
                         tpm_cell[[f"{name}_TPM_human_allele", f"{name}_TPM_chimp_allele"]], 
                         left_index=True, 
                         right_index=True, how='left')
    
    #add Total TPMs
    cell_total_tpm_table = endothelial_total_TPM.filter(like=total_tpm_column_indicator)
    cell_total_tpm_table.insert(cell_total_tpm_table.shape[1], f"{name}_TPM_total", cell_total_tpm_table.mean(axis=1))
    cell_info = pd.merge(cell_info, 
                         cell_total_tpm_table[[f"{name}_TPM_total"]], 
                         left_index=True, 
                         right_index=True, how='left')

    hybrids_dfs[name] = cell_info[[f'{name}_LFC_human_ref', 
                                   f'{name}_LFC_padj_human_ref', 
                                   f'{name}_LFC_chimp_ref', 
                                   f'{name}_LFC_padj_chimp_ref',
                                   f'{name}_TPM_total', 
                                   f"{name}_TPM_human_allele", 
                                   f"{name}_TPM_chimp_allele"]]

<a id="CO"></a>
## Cartilage Organoids

In [33]:
co_name = 'Cartilage_Organoids'
# human alignment
co_h_ref = pd.read_csv(import_path/f"Cartilage_Organoids/ASE/DESeq2_aligned2human.txt", sep = "\t")
co_h_ref.rename(columns={"gene": "Gene", 
                         "log2FoldChange": f"{co_name}_LFC_human_ref", 
                         "pvalue": f"{co_name}_pvalue_human_ref",
                         "padj": f"{co_name}_LFC_padj_human_ref", 
                         "padj_mine": f"{co_name}_padj_mine_human_ref", 
                         "qval_mine": f"{co_name}_qval_mine_human_ref",
                         "mean_ref_tpm": f"{co_name}_TPM_human_allele",
                         "mean_alt_tpm": f"{co_name}_TPM_chimp_allele",
                         "mean_total_tpm": f"{co_name}_TPM_total"}, inplace=True)

#chimp alignment
co_c_ref = pd.read_csv(import_path/f"Cartilage_Organoids/ASE/DESeq2_aligned2chimp.txt", sep = "\t")
co_c_ref.rename(columns={"gene": "Gene", 
                         "log2FoldChange": f"{co_name}_LFC_chimp_ref", 
                         "pvalue": f"{co_name}_pvalue_chimp_ref",
                         "padj": f"{co_name}_LFC_padj_chimp_ref", 
                         "padj_mine": f"{co_name}_padj_mine_chimp_ref", 
                         "qval_mine": f"{co_name}_qval_mine_chimp_ref"}, inplace=True)
# drop tpm columns from chimp ref to avoid duplication
co_c_ref = co_c_ref.drop(columns=['mean_ref_tpm', 'mean_alt_tpm', 'mean_total_tpm'])

# combine them
co_h_c = co_h_ref.merge(co_c_ref, how = "outer", on = "Gene").loc[:,["Gene",f"{co_name}_LFC_human_ref",
                                                                     f"{co_name}_LFC_padj_human_ref",
                                                                     f"{co_name}_LFC_chimp_ref",
                                                                     f"{co_name}_LFC_padj_chimp_ref",
                                                                     f"{co_name}_TPM_total",
                                                                     f"{co_name}_TPM_human_allele",
                                                                     f"{co_name}_TPM_chimp_allele"]]

co_h_c = co_h_c.set_index('Gene')

In [34]:
co_h_c.columns

Index(['Cartilage_Organoids_LFC_human_ref',
       'Cartilage_Organoids_LFC_padj_human_ref',
       'Cartilage_Organoids_LFC_chimp_ref',
       'Cartilage_Organoids_LFC_padj_chimp_ref',
       'Cartilage_Organoids_TPM_total', 'Cartilage_Organoids_TPM_human_allele',
       'Cartilage_Organoids_TPM_chimp_allele'],
      dtype='object')

In [35]:
hybrids_dfs[co_name] = co_h_c[[f'{co_name}_LFC_human_ref', 
                               f'{co_name}_LFC_padj_human_ref', 
                               f'{co_name}_LFC_chimp_ref', 
                               f'{co_name}_LFC_padj_chimp_ref',
                               f'{co_name}_TPM_total', 
                               f"{co_name}_TPM_human_allele", 
                               f"{co_name}_TPM_chimp_allele"]]

<a id="ExpLBM"></a>
# ExpLBM

In [36]:
lbm_name = 'ExpLBM'
# human alignment
lbm_h_ref = pd.read_csv(import_path/f"ExpLBM/outputs_05Jan2026_humanMPRA_draft1/ASE/DESeq2_aligned2human.txt", sep = "\t")
lbm_h_ref.rename(columns={"gene": "Gene", 
                         "log2FoldChange": f"{lbm_name}_LFC_human_ref", 
                         "pvalue": f"{lbm_name}_LFC_pvalue_human_ref",
                         "padj": f"{lbm_name}_LFC_padj_human_ref", 
                         "padj_mine": f"{lbm_name}_padj_mine_human_ref", 
                         "qval_mine": f"{lbm_name}_qval_mine_human_ref",
                         "mean_ref_tpm": f"{lbm_name}_TPM_human_allele", 
                         "mean_alt_tpm": f"{lbm_name}_TPM_chimp_allele",
                         "mean_total_tpm": f"{lbm_name}_TPM_total"}, inplace=True)

#chimp alignment
lbm_c_ref = pd.read_csv(import_path/f"ExpLBM/outputs_05Jan2026_humanMPRA_draft1/ASE/DESeq2_aligned2chimp.txt", sep = "\t")
lbm_c_ref.rename(columns={"gene": "Gene", 
                         "log2FoldChange": f"{lbm_name}_LFC_chimp_ref", 
                         "pvalue": f"{lbm_name}_LFC_pvalue_chimp_ref",
                         "padj": f"{lbm_name}_LFC_padj_chimp_ref", 
                         "padj_mine": f"{lbm_name}_padj_mine_chimp_ref", 
                         "qval_mine": f"{lbm_name}_qval_mine_chimp_ref",
                          "mean_ref_tpm": f"{lbm_name}_TPM_human_allele", 
                          "mean_alt_tpm": f"{lbm_name}_TPM_chimp_allele",
                          "mean_total_tpm": f"{lbm_name}_TPM_total"}, inplace=True)

# drop tpms df of chimp
lbm_c_ref = lbm_c_ref.drop(columns=[f"{lbm_name}_TPM_human_allele", 
                                    f"{lbm_name}_TPM_chimp_allele",
                                    f"{lbm_name}_TPM_total"])

# combine them
lbm_h_c = lbm_h_ref.merge(lbm_c_ref, how = "outer", on = "Gene").loc[:,["Gene",f"{lbm_name}_LFC_human_ref",
                                                                    f"{lbm_name}_LFC_pvalue_human_ref",
                                                                     f"{lbm_name}_LFC_padj_human_ref",
                                                                     f"{lbm_name}_LFC_chimp_ref",
                                                                     f"{lbm_name}_LFC_pvalue_chimp_ref",
                                                                     f"{lbm_name}_LFC_padj_chimp_ref",
                                                                     f"{lbm_name}_TPM_human_allele",
                                                                     f"{lbm_name}_TPM_chimp_allele",
                                                                     f"{lbm_name}_TPM_total"]]

lbm_h_c = lbm_h_c.set_index('Gene')

In [37]:
hybrids_dfs[lbm_name] = lbm_h_c[[f'{lbm_name}_LFC_human_ref',
                                 f'{lbm_name}_LFC_pvalue_human_ref',
                                 f'{lbm_name}_LFC_padj_human_ref', 
                                 f'{lbm_name}_LFC_chimp_ref',
                                 f'{lbm_name}_LFC_pvalue_chimp_ref',
                                 f'{lbm_name}_LFC_padj_chimp_ref',
                                 f'{lbm_name}_TPM_total', 
                                 f"{lbm_name}_TPM_human_allele", 
                                 f"{lbm_name}_TPM_chimp_allele"]]

<a id="combining"></a>
# Create combined hybrids data
And filter genes with no annotations

In [38]:
gene_meta_data = ipscs_cnccs_ASE_raw.set_index("Gene")[['Chromosome', 
                                                        'Start(hg38)', 
                                                        'End(hg38)', 
                                                        'Start(hg19)',
                                                        'End(hg19)',
                                                        'Strand', 
                                                        'ID(ENTREZ)', 
                                                        'Gene type', 
                                                        'ID(Ensemble)']]

In [39]:
# Separate old skin organoid cells (old clustering) from the rest
old_skin_keys = set(skin_cell_types) | set(pooled_cells.keys()) | {"Pooled_prechondral_mesenchyme"}
old_skin_dfs = {k: v.reset_index() for k, v in hybrids_dfs.items() if k in old_skin_keys}
new_hybrids_dfs = {k: v for k, v in hybrids_dfs.items() if k not in old_skin_keys}

# Build combined table without old skin organoids
hybrids_combined = reduce(lambda left, right: pd.merge(left,right, on=["Gene"], how='outer'), new_hybrids_dfs.values())
hybrids_combined = gene_meta_data.merge(hybrids_combined, left_index=True, right_index=True, how='outer')
hybrids_combined = hybrids_combined.reset_index()

# Build old skin organoids combined table (for separate export)
old_skin_combined = reduce(lambda left, right: pd.merge(left, right, on=["Gene"], how='outer'), old_skin_dfs.values())
old_skin_combined = gene_meta_data.merge(old_skin_combined, left_index=True, right_index=True, how='outer')
old_skin_combined = old_skin_combined.reset_index()

# Update hybrids_dfs so the ASE pipeline only processes new clustering cells
hybrids_dfs = new_hybrids_dfs

hybrids_combined.shape

(94180, 341)

In [40]:
hybrids_combined['ID(ENTREZ)'].isna().sum() 

48232

In [41]:
hybrids_combined_old = hybrids_combined.copy()

<b> Fill (some of the) missing NCBI IDs

In [42]:
NCBI_hg19 = pd.read_csv('/home/labs/davidgo/Collaboration/GenomeAnnotation/Human/NCBI/hg19/gff_parsed/NCBI_genes_hg19.txt', 
                        delimiter='\t')

In [43]:
human_symbol_to_id = dict(zip(hybrids_combined['Gene'], hybrids_combined['ID(ENTREZ)']))

In [44]:
NCBI_orthologs = pd.read_csv('/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Annotation DBs/NCBI Entrez Gene/orthologs/human_chimp.txt', sep='\t')
human_id_to_chimp_id = dict(zip(NCBI_orthologs['GeneID'], NCBI_orthologs['Other_GeneID']))

In [45]:
ncbi_genes_data = pd.read_csv('/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Annotation DBs/NCBI Entrez Gene/chimp/ncbi_genes_data.tsv', sep='\t')
chimp_id_to_symbol = dict(zip(ncbi_genes_data['Gene ID'], ncbi_genes_data['Symbol']))

In [46]:
chimp_id_to_symbol[human_id_to_chimp_id[26009]]

'ZZZ3'

In [47]:
mismatches = []

for human_symbol, human_id in human_symbol_to_id.items():
    chimp_id = human_id_to_chimp_id.get(human_id)
    if chimp_id is None:
        continue  # no chimp ortholog for this human gene id

    chimp_symbol = chimp_id_to_symbol.get(chimp_id)
    if chimp_symbol is None:
        continue  # chimp id has no symbol

    if chimp_symbol != human_symbol:
        mismatches.append((human_symbol, chimp_symbol, human_id, chimp_id))

print(f"Found {len(mismatches)} symbol mismatches.")
print("First 20:")
for row in mismatches[:20]:
    print(row)

Found 905 symbol mismatches.
First 20:
('AAED1', 'PRXL2C', 195827.0, 465250)
('AARS', 'AARS1', 16.0, 454210)
('ACPP', 'ACP3', 55.0, 460702)
('ADPRHL2', 'ADPRS', 54936.0, 456750)
('ADSS', 'ADSS2', 159.0, 457865)
('ADSSL1', 'ADSS1', 122622.0, 467568)
('AES', 'TLE5', 166.0, 744806)
('ALS2CR12', 'FLACC1', 130540.0, 459875)
('APOPT1', 'LOC736782', 84334.0, 736782)
('ARMC4', 'ODAD2', 55130.0, 450376)
('ARSE', 'ARSL', 415.0, 751059)
('ASAH2', 'LOC450461', 56624.0, 450461)
('ASNA1', 'GET3', 439.0, 455750)
('ATP5MD', 'ATP5MK', 84833.0, 735405)
('ATP5MPL', 'ATP5MJ', 9556.0, 453194)
('ATP5S', 'DMAC2L', 27109.0, 467450)
('B3GNT10', 'B3GALT9', 100288842.0, 100614645)
('BHLHB9', 'GPRASP3', 80823.0, 473708)
('BHMG1', 'MEIOSIN', 388553.0, 456137)
('BTBD11', 'ABTB3', 121551.0, 452206)


In [48]:
hybrids_combined = hybrids_combined.merge(NCBI_hg19[['GeneName', 'GeneID']], 
                                          left_on='Gene', 
                                          right_on='GeneName', 
                                          how='left', 
                                          suffixes=('', '_ref'))

In [49]:
hybrids_combined['ID(ENTREZ)'].isna().sum() 

48232

In [50]:
hybrids_combined['ID(ENTREZ)'] = hybrids_combined['ID(ENTREZ)'].fillna(hybrids_combined['GeneID'])
hybrids_combined = hybrids_combined.drop(columns=['GeneName', 'GeneID'])

In [51]:
hybrids_combined['ID(ENTREZ)'].isna().sum() 

40056

In [52]:
# Function to merge duplicate IDs
def merge_gene_rows(group):
    merged_row = group.ffill().bfill().iloc[0]  # Fill missing values
    if group["Chromosome"].isna().any():
        merged_row["Gene"] = group.loc[group["Chromosome"].isna(), "Gene"].values[0]
    return merged_row

In [53]:
merged_df = hybrids_combined.groupby("ID(ENTREZ)", group_keys=False).apply(merge_gene_rows).reset_index(drop=True)

In [54]:
merged_df['ID(ENTREZ)'].isna().sum() 

0

Adding back the gene symbols with NaN in ID(ENTREZ) (that were removed when applying merge_gene_rows)

In [55]:
hybrids_combined = pd.concat([merged_df, hybrids_combined[hybrids_combined['ID(ENTREZ)'].isna()]])

In [56]:
hybrids_combined['ID(ENTREZ)'].isna().sum() 

40056

In [61]:
# # some genes in the ASE file dont have coordinates in hg19 so we dont define them as ASE and dont use them for windows
# hybrids_combined_filtered = hybrids_combined[~(pd.isna(hybrids_combined['Start(hg19)']) | pd.isna(hybrids_combined['End(hg19)']))].copy()
# print("There are {} genes without Hg19 coordinates".format(len(hybrids_combined) - len(hybrids_combined_filtered)))

In [62]:
# biomart_gene_ids = pd.read_csv('/home/labs/davidgo/Collaboration/GenomeAnnotation/Human/Ensembl_Biomart/ensembl_genes.txt', 
#                         delimiter='\t')
# symbol_to_ncbi = biomart_gene_ids.set_index('Gene name')['NCBI gene (formerly Entrezgene) ID'].to_dict()
# hybrids_combined["ID(ENTREZ)"] = hybrids_combined["ID(ENTREZ)"].fillna(hybrids_combined["Gene"].map(symbol_to_ncbi))
# hybrids_combined['ID(ENTREZ)'].isna().sum() 

<a id="adding_tss"></a>
### Adding TSS info

In [63]:
# Adding TSS information for each gene (according to strand(direction))
tss_values = np.where(hybrids_combined.Strand == '+', 
                      hybrids_combined['Start(hg19)'], 
                      hybrids_combined['End(hg19)'])

# Insert the TSS column at the desired location (e.g., position 2)
hybrids_combined.insert(9, 'TSS', tss_values)

In [64]:
hybrids_combined.shape

(92618, 342)

<a id="create_ase"></a>
# Create ASE Data

<a id="define_ASE"></a>
## Define ASE Genes
To define ASE, nonASE and rest genes we look of FC and adjusted P-value value in aligment to chimp and humans
(4 columns: LFC_chimp, padj_chimp, LFC_human, padj_human) and those values need to fit these rules:
1. No NA in those 4 columns
2. In both alignments the expression change is in the same direction - both LFC < 0 or both LFC > 0
3. The difference between the 2 LFC values will be smaller than 1

<b>For ASE:</b> also both padj values smaller than 0.05 </br>
<b>For nonASE:</b> one or both of the padj values are bigger or equal to 0.05 </br>
<b>For rest:</b> genes that are not in ASE or nonASE genes </br>

<b> Those definitions are done seperatly for each cell type and are stored in a dictionary named ase_data_dict in this format: {cell_type : [ASE_df,non_ASE_df,rest_df]} </b>

In [65]:
hybrids_data_to_use = hybrids_combined.copy()

In [66]:
# new pipeline
ase_data_dict_new = {}
ase_count_per_cell = {}

for cell in hybrids_dfs.keys():
    # Create masks for each filtering criterion
    na_mask = (pd.isna(hybrids_data_to_use[f'{cell}_LFC_human_ref']) |
               pd.isna(hybrids_data_to_use[f'{cell}_LFC_padj_human_ref']) |
               pd.isna(hybrids_data_to_use[f'{cell}_LFC_chimp_ref']) |
               pd.isna(hybrids_data_to_use[f'{cell}_LFC_padj_chimp_ref']))
    
    non_significant_mask = (hybrids_data_to_use[f"{cell}_LFC_padj_human_ref"] > 0.05) | \
                           (hybrids_data_to_use[f"{cell}_LFC_padj_chimp_ref"] > 0.05)
                     
    non_agreement_mask = (hybrids_data_to_use[f"{cell}_LFC_human_ref"] *
                            hybrids_data_to_use[f"{cell}_LFC_chimp_ref"] < 0)
                      
    non_close_value_mask = (hybrids_data_to_use[f"{cell}_LFC_human_ref"] - 
                            hybrids_data_to_use[f"{cell}_LFC_chimp_ref"]).abs() >= 1
        
    # Calculate the final masks for each category
    rest_mask = na_mask | non_agreement_mask | non_close_value_mask
    non_ase_mask = (~rest_mask) & (non_significant_mask)
    ase_mask = ~(rest_mask | non_ase_mask)
    
    # Extract genes for each category
    ASE_genes = hybrids_data_to_use[ase_mask]
    nonASE_genes = hybrids_data_to_use[non_ase_mask]
    rest_genes = hybrids_data_to_use[rest_mask]
    
    # Define columns to use
    columns_to_use = ['Gene', 
                      'Chromosome', 
                      'Start(hg38)', 
                      'End(hg38)', 
                      'Start(hg19)', 
                      'End(hg19)', 
                      'Strand', 
                      'ID(ENTREZ)', 
                      'Gene type', 
                      'ID(Ensemble)',
                      'TSS', 
                      f'{cell}_TPM_total',
                      f'{cell}_TPM_human_allele', 
                      f'{cell}_TPM_chimp_allele',
                      f'{cell}_LFC_human_ref', 
                      f'{cell}_LFC_padj_human_ref',
                      f'{cell}_LFC_chimp_ref', 
                      f'{cell}_LFC_padj_chimp_ref']
    
    # Store the data in the dictionary
    ase_data_dict_new[cell] = [
        ASE_genes.reindex(columns=columns_to_use),
        nonASE_genes.reindex(columns=columns_to_use),
        rest_genes.reindex(columns=columns_to_use)
    ]
    
    ase_count_per_cell[cell] = {"ASE": len(ASE_genes),
                                "nonASE": len(nonASE_genes),
                                "rest": len(rest_genes)}
    
    print(f"{cell} ASE genes: {len(ASE_genes)}")
    print(f"{cell} nonASE genes: {len(nonASE_genes)}")
    print(f"{cell} rest of genes: {len(rest_genes)}")
    print("_____" * 8)

AllFibroblasts ASE genes: 4767
AllFibroblasts nonASE genes: 4754
AllFibroblasts rest of genes: 83097
________________________________________
AllFollicle ASE genes: 1020
AllFollicle nonASE genes: 4624
AllFollicle rest of genes: 86974
________________________________________
AllKeratinocytes ASE genes: 3712
AllKeratinocytes nonASE genes: 5241
AllKeratinocytes rest of genes: 83665
________________________________________
APOE+_PTGDS+_lipid-associated_mesenchyme ASE genes: 168
APOE+_PTGDS+_lipid-associated_mesenchyme nonASE genes: 1466
APOE+_PTGDS+_lipid-associated_mesenchyme rest of genes: 90984
________________________________________
ATP1B3+_DPSYL2+_basal_keratinocytes ASE genes: 757
ATP1B3+_DPSYL2+_basal_keratinocytes nonASE genes: 2435
ATP1B3+_DPSYL2+_basal_keratinocytes rest of genes: 89426
________________________________________
Cycling_follicular_progenitors ASE genes: 110
Cycling_follicular_progenitors nonASE genes: 1114
Cycling_follicular_progenitors rest of genes: 91394
______

<a id="create_ase_table"></a>
## Create Gene ASE information Table 
Create a table with gene information for each of the cell types in ase_data_dict dictionary (which gene is considered ASE or nonASE)

In [67]:
def is_ASE(row, ASE_genes_list, nonASE_genes_list, gene_ID_column):
    ''' 
    row - a row in a df (using apply)
    ASE_genes - a list of genes that are considered ASE in the chosen cell type
    nonASE_genes - a list of genes that are considered nonASE in the chosen cell type
    gene_ID_column - the column in the df that contains the ID of the gene to search in ASE genes IDs
    output - returns "ASE" if the gene in ASE_genes_list, "nonASE" if the gene in nonASE_genes_list and "else" otherwise.
    '''
    if row[gene_ID_column] in ASE_genes_list:
        return "ASE"
    elif row[gene_ID_column] in nonASE_genes_list:
        return "nonASE"
    return "else"

In [68]:
gene_is_ase_info_new = hybrids_data_to_use.loc[:,['Gene', 
                                       'Chromosome', 
                                       'Start(hg38)',
                                       'End(hg38)', 
                                       'Start(hg19)', 
                                       'End(hg19)', 
                                       'Strand', 
                                       'ID(ENTREZ)', 
                                       'Gene type', 
                                       'ID(Ensemble)']].copy()

In [69]:
# adding gene_ase_type column (if a gene is ASE or nonASE)
for cell_type in ase_data_dict_new.keys():
    column_name = f"{cell_type}_gene_ase_type"
    ASE_genes = list(ase_data_dict_new[cell_type][0]["ID(ENTREZ)"])
    nonASE_genes = list(ase_data_dict_new[cell_type][1]["ID(ENTREZ)"])
    
    gene_is_ase_info_new[column_name] = gene_is_ase_info_new.apply(is_ASE,
                                                                   axis = 1, 
                                                                   args = (ASE_genes, nonASE_genes, "ID(ENTREZ)"))

In [70]:
# add a column that counts for each gene in how many cell types it is considered ASE 
gene_is_ase_info_new["ASE_count"] = gene_is_ase_info_new.iloc[:,10:].isin(["ASE"]).sum(axis=1)

In [71]:
ase_info_table = hybrids_data_to_use.set_index('Gene').merge(gene_is_ase_info_new.set_index('Gene').iloc[:,9:], 
                                                             left_index=True, 
                                                             right_index=True)

In [72]:
# What cells currently have TPM values
ase_info_table.filter(like='TPM_total').dropna(axis=1, how='all').columns

Index(['Motor_neuron_(MN)_TPM_total',
       'Retinal_pigment_epithelium_(RPE)_TPM_total',
       'Skeletal_myocyte_(SKM)_TPM_total', 'Cardiomyocyte_(CM)_TPM_total',
       'Hepatocyte_Progenitor_(HP)_TPM_total',
       'Pancreatic_progenitor_(PP)_TPM_total', 'iPSCs_TPM_total',
       'CNCCs_TPM_total', 'Cortical_organoids_d50_TPM_total',
       'Cortical_organoids_d100_TPM_total',
       'Cortical_organoids_d150_TPM_total',
       'Endothelial_not_treated_TPM_total', 'Endothelial_TNF_TPM_total',
       'Endothelial_Spin_TPM_total', 'Cartilage_Organoids_TPM_total',
       'ExpLBM_TPM_total'],
      dtype='object')

## Create sub groups for reduced vs induced expression

In [73]:
def split_ase_to_reduced_induced(cell_type, cell_ase_df):
    reduced_cell_ase_df = cell_ase_df[(cell_ase_df[f'{cell_type}_LFC_human_ref'] < 0) & 
                                         (cell_ase_df[f'{cell_type}_LFC_chimp_ref'] < 0)]
    induced_cell_ase_df = cell_ase_df[(cell_ase_df[f'{cell_type}_LFC_human_ref'] > 0) & 
                                         (cell_ase_df[f'{cell_type}_LFC_chimp_ref'] > 0)]
    
    return reduced_cell_ase_df, induced_cell_ase_df

In [74]:
induced_ase_data_dict = {}
reduced_ase_data_dict = {}

for cell in ase_data_dict_new.keys():
    reduced_cell_ase_df, induced_cell_ase_df = split_ase_to_reduced_induced(cell, ase_data_dict_new[cell][0])
    
    reduced_ase_data_dict[cell] = [
       reduced_cell_ase_df,
       ase_data_dict_new[cell][1],
       ase_data_dict_new[cell][2]
   ]
    
    induced_ase_data_dict[cell] = [
       induced_cell_ase_df,
       ase_data_dict_new[cell][1],
       ase_data_dict_new[cell][2]
   ]

## Reorder columns

In [75]:
# TODO

In [76]:
# ['Pooled_fibroblasts',
#  '0_Fibroblast',
#  '1_Fibroblast',
#  '3_Fibroblast',
#  '5_Fibroblast',
#  '10_Fibroblast',
 
#  'Pooled_epidermis',
#  'Cycling_Keratinocyte_Precursors',
#  'HF_Inner_Root_Sheath',
#  'HF_Outer_Root_Sheath',
#  'HF_Sebocytes',
#  'KRT6B+_Root_Sheath',
 
#  'Pooled_mesenchyme',
#  '0_Fibroblast',
#  '1_Fibroblast',
#  '2_Mesenchyme',
#  '3_Fibroblast',
#  '4_Prechondral_Mesenchyme',
#  '5_Fibroblast',
#  '10_Fibroblast',
#  '11_Weird_CNC_Fibroblast',
#  '7_Fibroblast_Mesenchyme_Mash',
#  '8_Also_Prechondral_Mesenchyme',
 
#  'Pooled_prechondral_mesenchyme'
#  '4_Prechondral_Mesenchyme',
#  '8_Also_Prechondral_Mesenchyme',
 
 
#  '6_Dermal_Papilla',
#  '9_Chondrocytes',
#  '10_Neural_Progenitors_+_BAD'
#  '12_Schwann_Cells',
#  '16_Melanocytes',
#  '18_Proliferating_Cells',
#  '20_Absolutely_No_Idea',
#  '21_Merkel_Cells',
#  'Keratinocytes',
#  'Motor_neuron_(MN)',
#  'Retinal_pigment_epithelium_(RPE)',
#  'Skeletal_myocyte_(SKM)',
#  'Cardiomyocyte_(CM)',
#  'Hepatocyte_Progenitor_(HP)',
#  'Pancreatic_progenitor_(PP)',
#  'IPS',
#  'CNCC',
#  'brain100',
#  'endothelial_not_treated',
#  'endothelial_TNF',
#  'endothelial_Spin']

<a id="export_ase"></a>
## Export

### Export table

In [77]:
pd.DataFrame.from_dict(ase_count_per_cell, orient='index').to_csv(export_path/'ase_type_counts_per_cell.tsv', sep='\t')

In [78]:
ase_info_table.to_csv(export_path/'ASE_info.tsv', sep='\t')
old_skin_combined.to_csv(export_path/'old_skin_organoids_clustering_ASE_info.tsv', sep='\t')

### Export pickle

In [79]:
with open(export_path/'ASE_info.pickle', 'wb') as handle:
   pickle.dump(ase_data_dict_new, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(export_path/'reduced_ASE_info.pickle', 'wb') as handle:
   pickle.dump(reduced_ase_data_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(export_path/'induced_ASE_info.pickle', 'wb') as handle:
   pickle.dump(induced_ase_data_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

<a id="bed"></a>
## Write a bed file for UCSC tracks
for all ASE and nonASE genes in all cell types

In [ ]:
# # write a bed file for UCSC tracks for all ase and nonase genes in all cell types
# # Open a file to write the BED content

# with open(import_path/"UCSC_tracks/ASE_and_nonASE_genes_tracks.bed", "w") as bed_file:
#     for cell_type, (df_ase, df_nonase, df_else) in ase_data_dict.items():
        
#         # Write track definition for ASE genes
#         bed_file.write(f'track name="{cell_type} ASE genes" description="{cell_type} ASE genes" color=255,0,0\n')
#         # Write ASE gene TSS locations
#         for index, row in df_ase.iterrows():
#             bed_file.write(f"{row['Chromosome']}\t{int(row['TSS']-1)}\t{int(row['TSS'])}\tASE_{cell_type}_{int(row['ID(ENTREZ)'])}\n")

#         # Write track definition for nonASE genes
#         bed_file.write(f'track name="{cell_type} nonASE genes" description="{cell_type} nonASE genes" color=128,128,128\n')
#         # Write nonASE gene TSS locations
#         for index, row in df_nonase.iterrows():
#             bed_file.write(f"{row['Chromosome']}\t{int(row['TSS']-1)}\t{int(row['TSS'])}\tNonASE_{cell_type}_{int(row['ID(ENTREZ)'])}\n")